In [17]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [18]:
files = ["30_data/46343_30_dataset.csv", "30_data/759667_30_dataset.csv",
         "30_data/781756_30_dataset.csv", "30_data/844359_30_dataset.csv",
         "30_data/1066528_30_dataset.csv", "30_data/1360686_30_dataset.csv",
         "30_data/1449548_30_dataset.csv"]

test_files = ["30_data/1455390_30_dataset.csv", "30_data/9618981_30_dataset.csv",]

In [19]:
df_list = [pd.read_csv(file) for file in files]

combined_df = pd.concat(df_list, ignore_index=True)

In [20]:
combined_df

,epoch_id,sleep_stage,mean_rr,std_rr,rr_norm_diff,rr_delta,mean_enmo,max_enmo,std_enmo
0,0,-1,632.619161,11.966930,0.000000,0.000000,0.007280,0.312825,0.017407
1,1,-1,682.274764,31.027475,47.172823,49.655603,0.006448,0.103841,0.013130
2,2,-1,744.957807,20.824294,104.363072,62.683043,0.006008,0.111695,0.013064
3,3,-1,723.368035,8.047318,78.634636,-21.589772,0.002392,0.081855,0.005852
4,4,-1,720.164502,12.908450,71.659547,-3.203533,0.006515,0.093086,0.013309
...,...,...,...,...,...,...,...,...,...
3978,960,0,927.033908,9.015589,15.602639,-10.784524,0.002451,0.005710,0.000914
3979,961,0,920.278370,5.685253,8.404746,-6.755538,0.002430,0.005852,0.000914
3980,962,0,943.140471,26.095342,29.703504,22.862101,0.002421,0.006389,0.000931
3981,963,0,966.689162,16.613707,50.589586,23.548692,0.002438,0.006904,0.000978


In [21]:
test_df = pd.read_csv(test_files[0])

In [22]:
features = ['rr_norm_diff', 'rr_delta', 'mean_enmo', 'max_enmo', 'std_enmo']
label = "sleep_stage"

In [23]:
# лейбли сну не підряд 5->4
combined_df[label] = combined_df[label].replace({5: 4})
test_df[label] = test_df[label].replace({5: 4})

In [24]:
#кастиль, перевірити!
combined_df = combined_df[combined_df[label] >= 0]
test_df = test_df[test_df[label] >= 0]

In [25]:
X_train = combined_df[list(features)]
Y_train = combined_df[label]

In [26]:
X_test = test_df[features]
Y_test = test_df[label]

In [27]:
print(f"Розмір тренувальної вибірки: {X_train.shape}")
print(f"Розмір тестової вибірки: {X_test.shape}")

Розмір тренувальної вибірки: (3915, 5)
Розмір тестової вибірки: (670, 5)


In [28]:
model = RandomForestClassifier(
    n_estimators=200,         # Кількість дерев у лісі
    max_depth=7,              # Глибина дерева (трохи глибше, ніж в XGBoost)
    class_weight='balanced',  # Ідеально балансує рідкісний N1 та частий N2
    n_jobs=-1,                # Використати всі потужні ядра твого M4!
    random_state=42
)

In [29]:
model.fit(X_train, Y_train)

RandomForestClassifier(class_weight='balanced', max_depth=7, n_estimators=200,
                       n_jobs=-1, random_state=42)

In [30]:
predictions = model.predict(X_test)

In [31]:
print("\n--- Результати на тестовому юзері ---")
print(f"Загальна точність (Accuracy): {accuracy_score(Y_test, predictions):.3f}")
print("\nДетальний звіт по фазах сну:")
target_names = ['Wake', 'N1', 'N2', 'N3', 'REM']
#target_names = ['T', 'F']
print(classification_report(Y_test, predictions, target_names=target_names))

# 5. Дивимося важливість фічів
importance = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\nВажливість ознак:")
print(importance)


--- Результати на тестовому юзері ---
Загальна точність (Accuracy): 0.375

Детальний звіт по фазах сну:
              precision    recall  f1-score   support

        Wake       0.23      0.63      0.34        70
          N1       0.00      0.00      0.00        42
          N2       0.85      0.33      0.47       490
          N3       0.00      0.00      0.00         0
         REM       0.18      0.69      0.28        68

    accuracy                           0.37       670
   macro avg       0.25      0.33      0.22       670
weighted avg       0.66      0.37      0.41       670


Важливість ознак:
        Feature  Importance
0  rr_norm_diff    0.244260
2     mean_enmo    0.234674
4      std_enmo    0.186984
3      max_enmo    0.178089
1      rr_delta    0.155993


/Users/anastasiiazabolotna/Documents/lpnu/4sem/command/Smart-Alarm-work/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/anastasiiazabolotna/Documents/lpnu/4sem/command/Smart-Alarm-work/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/anastasiiazabolotna/Documents/lpnu/4sem/command/Smart-Alarm-work/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use 